<a href="https://colab.research.google.com/github/sweetscoding/belajarMLfrom-Zero/blob/main/01_Web_Scraping_Starter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **1. Install Library**

Pada tahap ini, kita melakukan instalasi beberapa library yang dibutuhkan.

In [10]:
!pip install requests
!pip install beautifulsoup4

## **2. Import Library**

Untuk menggunakan library pada Python, kita harus melakukan import library tersebut menggunakan perintah `import`.

In [11]:
import requests
from bs4 import BeautifulSoup
import time
import pandas as pd

## **3. Fungsi Scrap City (`ambil_data`)**

Fungsi `scrap_city` berfungsi untuk mengambil data-data rumah pada kota tertentu.

In [67]:
def ambil_data(city,url,page=1):
  print(f"Ambil data halaman {page} untuk kota {city}")

  headers ={
      "User-Agent": "Mozilla/5.0 (Windows NT 10.0: Wind64; x64)"
  }

  #ambil konten html pada url tersebut
  response = requests.get(url + f"?page={page}", headers=headers)

  #cek response status code
  print(response.status_code)

  # Handle 429 (Too Many Requests) explicitly
  if response.status_code == 429:
    print(f"Warning: Received 429 (Too Many Requests) for {city} on page {page}. Waiting for 10 seconds and skipping this page.")
    time.sleep(10) # Wait longer if rate-limited
    return [] # Return empty list for this page

  # If not 200, but also not 429, it could be other errors. Print content and return empty.
  if response.status_code != 200:
    print(f"Error: Received status code {response.status_code} for {city} on page {page}. Skipping this page.")
    print("Dumping first 500 characters of response text:\n")
    print(response.text[:500])
    return []

  #buat objek BeautifulSoup
  soup = BeautifulSoup(response.text, "html.parser")

  #ambil container utama
  container = soup.find('div', class_='card-list-section')

  if container is None:
    print(f"Error: Could not find the main container with class 'card-list-section' for {city} on page {page}. The page content might have changed or been empty.")
    return [] # Return an empty list if container not found to prevent further errors

  # ambil elemen rumah
  featured = container.find_all('div', class_='featured-card-component')

  print(f"Jumlah rumah: {len(featured)}")

  data = [] #simpan data scraping

  #loop untuk setiap rumah
  for idx , house in enumerate(featured):
    #ambil container untuk rumah
    content = house.find('div', class_='card-featured__middle-section')

    if content is None:
      continue

    #ambil data harga
    price_element = content.find('div', class_='card-featured__middle-section__price')
    price = price_element.text.strip() if price_element else ""

    #ambil judul rumah
    # Check content.contents length before accessing
    title = content.contents[2].text.strip() if len(content.contents) > 2 else ""

    #ambil lokasi rumah
    location = content.contents[3].text.strip() if len(content.contents) > 3 else ""

    #ambil container untuk jml kamar, kmr mandi, garasi
    features_container = content.find('div', class_='card-featured__middle-section__attribute')

    bedroom = ""
    bathroom = ""
    garage = ""
    area = ""
    building_area = ""

    if features_container:
      #ambil span yang menjadi root tag
      attributes = features_container.find_all('span', class_='attribute-text')

      if len(attributes) > 0:
        bedroom = attributes[0].text.strip()
      if len(attributes) > 1:
        bathroom = attributes[1].text.strip()
      if len(attributes) > 2:
        garage = attributes[2].text.strip()

      #ambil luas tanah dan bangunan
      # Ensure there are enough contents to avoid IndexError
      if len(features_container.contents) > 1:
        area = features_container.contents[1].text.strip()
      if len(features_container.contents) > 2:
        building_area = features_container.contents[2].text.strip()

    # Add scraped data to the list
    data.append({
        "city": city,
        "title": title,
        "location": location,
        "price": price,
        "bedroom": bedroom,
        "bathroom": bathroom,
        "garage": garage,
        "area": area,
        "building_area": building_area
    })

  time.sleep(3)
  return data

## **4. Fungsi untuk Mulai Scrap**

Fungsi `start_scrap` berfungsi untuk memulai proses scraping pada beberapa kota. Didalamnya, fungsi ini akan memanggil fungsi `ambil_data`.

In [68]:
def start_scrap(cities, max_page_per_city =10):
  data = []

  for city, url in cities.items():
    for page in range(1,max_page_per_city+1):
      page_data = ambil_data(city,url,page)
      data.extend(page_data)

  return data

## **5. Program Utama**

Ini adalah program utama. Program akan dimulai pada kode ini.

In [73]:
cities = {
    "Magelang": "https://www.rumah123.com/jual/magelang/rumah/"
}

#tes fungsi ambil_data
# ambil_data("Magelang","https://www.rumah123.com/jual/magelang/rumah/")


#fungsi mulai scrap
data = start_scrap(cities,1)

#simpan dalam csv
df = pd.DataFrame(data)
df.to_csv("rumah123.csv", index=False)

Ambil data halaman 1 untuk kota Magelang
200
Jumlah rumah: 20
